In [4]:
# Install AutoGen Studio and pyngrok
!pip install -q autogenstudio pyngrok

# Verify the installation
!autogenstudio version

AutoGen Studio  CLI version: 0.4.2.2


In [5]:
import os
import getpass

# 1. Set up Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Set up Ngrok Auth Token (Required to tunnel the AutoGen Studio UI)
NGROK_TOKEN = getpass.getpass("Enter your Ngrok Auth Token: ")
!ngrok config add-authtoken {NGROK_TOKEN}

Authtoken saved to configuration file: C:\Users\Administrator\AppData\Local\ngrok\ngrok.yml


In [6]:
import multiprocessing
import time
from pyngrok import ngrok

def run_autogen_studio():
    # Launch AutoGen Studio on port 8081
    !autogenstudio ui --port 8081 --host 0.0.0.0

# Start AutoGen Studio in a background process
process = multiprocessing.Process(target=run_autogen_studio)
process.start()

# Wait a moment for the server to spin up
time.sleep(5)

# Open the ngrok tunnel to port 8081
public_url = ngrok.connect(8081)
print("\n" + "="*60)
print(f"[SUCCESS] AutoGen Studio is running!")
print(f"Click the link below to open the UI:")
print(f"{public_url}")
print("="*60 + "\n")


[SUCCESS] AutoGen Studio is running!
Click the link below to open the UI:
NgrokTunnel: "https://energetic-cardboard-smock.ngrok-free.dev" -> "http://localhost:8081"



In [7]:
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def run_team():
    groq_key = os.environ.get("GROQ_API_KEY", "")

    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."
    )

    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."
    )

    team = RoundRobinGroupChat(
        participants=[researcher, editor],
        termination_condition=MaxMessageTermination(max_messages=4)
    )

    print("--- Starting Multi-Agent Session ---")
    async for message in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):
        # Skip the final TaskResult object — only print agent messages
        if hasattr(message, "source"):
            print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
            print("-" * 40)

await run_team()

--- Starting Multi-Agent Session ---

[user]: Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs.
----------------------------------------

[Researcher]: **Groq LPUs: High-Throughput Processors for Large Language Models (LLMs)**

**Introduction**
---------------

Groq's LPUs (Lookup Tables) are highly parallelized processors designed to accelerate Large Language Models (LLMs) and other machine learning workloads. LPUs have emerged as a promising alternative to traditional GPUs for LLM training and inference tasks. In this section, we will delve into the architecture and design decisions behind Groq LPUs and explore why they provide higher throughput for LLMs compared to standard GPUs.

**LPU Architecture**
-------------------

### Customized Design

Groq LPUs are custom-designed processors tailored for LLM workloads. This optimized architecture differs significantly from general-purpose CPUs and GPUs. LPUs are based on a systolic array architecture, which allow

In [ ]:
# ============================================================
#                    CELL 5 - EXTENSION TASKS
# ============================================================
# This cell extends the basic 2-agent system from Cell 4
# with 2 additional features:
#
# EXTENSION 1: Tool-Wielding Agent
#   - Added a real-world stock price fetcher tool (Python function)
#   - Attached it to the Researcher agent
#   - Task forces Researcher to CALL the tool to get live data
#   - Groq's function-calling capability handles this natively
#
# EXTENSION 2: UserProxy (Human-in-the-Loop)
#   - Added a UserProxyAgent as the 3rd participant
#   - After every Researcher + Editor round, conversation PAUSES
#   - YOU type feedback or approval in the input box
#   - Type "TERMINATE" to end the session cleanly
#
# AGENT FLOW:
#   Researcher (tool call) → Editor (critique) → UserProxy (you)
#   → repeats until TERMINATE or max 6 messages
# ============================================================

import os
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

# ============================================================
# EXTENSION 1 — TOOL DEFINITION
# A real-world stock price fetcher using Yahoo Finance
# No API key needed — uses public endpoint
# The Researcher agent will CALL this function automatically
# when asked about stock prices
# ============================================================
def get_stock_price(ticker: str) -> str:
    """Fetches the latest stock price for a given ticker symbol."""
    try:
        import urllib.request
        import json
        # Yahoo Finance public API — no authentication needed
        url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}?interval=1d&range=1d"
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read())
            price = data["chart"]["result"][0]["meta"]["regularMarketPrice"]
            currency = data["chart"]["result"][0]["meta"]["currency"]
            return f"{ticker} current price: {price} {currency}"
    except Exception as e:
        return f"Error fetching stock price: {str(e)}"


# ============================================================
# MAIN ASYNC FUNCTION — runs the full multi-agent pipeline
# ============================================================
async def run_team():

    # --------------------------------------------------------
    # Fetch Groq API key set securely in Cell 2
    # --------------------------------------------------------
    groq_key = os.environ.get("GROQ_API_KEY", "")

    # --------------------------------------------------------
    # RESEARCHER MODEL — faster, lightweight Groq model
    # Used for research + tool calling
    # --------------------------------------------------------
    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,   # needed for tool calling
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # --------------------------------------------------------
    # EDITOR MODEL — larger, more capable Groq model
    # Used for critique and professional refinement
    # --------------------------------------------------------
    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # --------------------------------------------------------
    # EXTENSION 1 — RESEARCHER AGENT WITH TOOL
    # tools=[get_stock_price] attaches the real-world fetcher
    # System message explicitly instructs it to USE the tool
    # --------------------------------------------------------
    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        tools=[get_stock_price],  # <-- tool attached here
        system_message="""You are an expert researcher with access to real-world tools.
        When asked about stock prices, you MUST call the get_stock_price tool first.
        Then provide a detailed analysis in clear Markdown format."""
    )

    # --------------------------------------------------------
    # EDITOR AGENT — no tool needed
    # Reviews researcher output and asks UserProxy for approval
    # --------------------------------------------------------
    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="""You are a strict editor.
        Review the researcher's findings and optimize for professional delivery.
        After your review, ask the UserProxy for approval before finishing."""
    )

    # --------------------------------------------------------
    # EXTENSION 2 — USER PROXY AGENT (Human-in-the-Loop)
    # input_func=input means it ALWAYS pauses for your response
    # You can type feedback or type TERMINATE to stop
    # --------------------------------------------------------
    user_proxy = UserProxyAgent(
        name="UserProxy",
        input_func=input  # pauses and waits for YOUR input every turn
    )

    # --------------------------------------------------------
    # TEAM SETUP — RoundRobin means agents go in fixed order:
    # Researcher → Editor → UserProxy → repeat
    # Stops at 6 messages OR when you type TERMINATE
    # --------------------------------------------------------
    team = RoundRobinGroupChat(
        participants=[researcher, editor, user_proxy],
        termination_condition=MaxMessageTermination(max_messages=6)
    )

    # --------------------------------------------------------
    # SESSION START — instructions printed for the user
    # --------------------------------------------------------
    print("=" * 55)
    print("   EXTENSION TASK — Multi-Agent Session Started")
    print("=" * 55)
    print("AGENT ORDER: Researcher → Editor → UserProxy (You)")
    print("-" * 55)
    print("💡 When UserProxy turn arrives:")
    print("   → Type your feedback and press Enter to continue")
    print("   → Type TERMINATE and press Enter to end session")
    print("=" * 55 + "\n")

    # --------------------------------------------------------
    # TASK — forces Researcher to call get_stock_price tool
    # This is the real-world data trigger for Extension 1
    # --------------------------------------------------------
    async for message in team.run_stream(
        task="Find the current stock price of Apple (AAPL) and provide a brief analysis of what it means for investors."
    ):
        # Skip the final TaskResult object — only print agent messages
        if hasattr(message, "source"):
            print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
            print("-" * 40)

# ============================================================
# RUN — executes the async function inside Google Colab
# ============================================================
await run_team()

   EXTENSION TASK — Multi-Agent Session Started
AGENT ORDER: Researcher → Editor → UserProxy (You)
-------------------------------------------------------
💡 When UserProxy turn arrives:
   → Type your feedback and press Enter to continue
   → Type TERMINATE and press Enter to end session


[user]: Find the current stock price of Apple (AAPL) and provide a brief analysis of what it means for investors.
----------------------------------------

[Researcher]: [FunctionCall(id='43p6jj31d', arguments='{"ticker":"AAPL"}', name='get_stock_price')]
----------------------------------------

[Researcher]: [FunctionExecutionResult(content='AAPL current price: 298.01 USD', name='get_stock_price', call_id='43p6jj31d', is_error=False)]
----------------------------------------

[Researcher]: AAPL current price: 298.01 USD
----------------------------------------

[Editor]: **Editor's Review**

The current stock price of Apple (AAPL) is $298.01 USD. To provide a brief analysis, this price point sugges

t=2026-06-22T15:11:09+0530 lvl=warn msg="failed to open private leg" id=33574daccfc5 privaddr=localhost:8081 err="dial tcp [::1]:8081: connectex: No connection could be made because the target machine actively refused it."
t=2026-06-22T15:12:19+0530 lvl=warn msg="failed to open private leg" id=775dec6a34d8 privaddr=localhost:8081 err="dial tcp [::1]:8081: connectex: No connection could be made because the target machine actively refused it."



[UserProxy]: is this good time to buy apple stock based on the current price
----------------------------------------

[Researcher]: [FunctionCall(id='nraeqdhhn', arguments='{"ticker":"AAPL"}', name='get_stock_price')]
----------------------------------------

[Researcher]: [FunctionExecutionResult(content='AAPL current price: 298.01 USD', name='get_stock_price', call_id='nraeqdhhn', is_error=False)]
----------------------------------------

[Researcher]: AAPL current price: 298.01 USD
----------------------------------------

[Editor]: **Editor's Review**

Based on the current price of $298.01 USD, whether it is a good time to buy Apple (AAPL) stock depends on various factors, including your individual investment goals, risk tolerance, and market outlook. Here's a brief analysis:

**Technical Analysis:**

1. **Trend Analysis**: Apple's stock has historically demonstrated a strong upward trend, with periodic fluctuations. The current price of $298.01 USD may represent a relatively sta

t=2026-06-22T16:41:00+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=70015c49c969 err="read EOF from remote peer"
t=2026-06-22T16:41:02+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=815f2e0cd471 clientid=30df56e9db8d27e7a3f5058dc736a220
t=2026-06-22T17:47:18+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=fc5f1364f6b2 clientid=30df56e9db8d27e7a3f5058dc736a220
t=2026-06-22T17:47:18+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=70015c49c969 err="session closed"
t=2026-06-22T19:47:28+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=62e3ebd66d10 clientid=30df56e9db8d27e7a3f5058dc736a220
t=2026-06-22T19:47:28+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=70015c49c969 err="session closed"
t=2026-06-22T19:47:38+0530 lvl=eror